# **HTGNN FOREX XAI Workflow**

This notebook shows the reproducible workflow for the XAI case study: data preparation, model training, backtesting, and explanation visualisation. Command cells use `!` so they run in the terminal from inside Jupyter.

## 1. Data preparation

Run these cells when starting from an empty `data/` directory. The first command downloads Yahoo Finance market data; the second transforms it into supervised tensors and heterogeneous node windows.

In [ ]:
!python -m src.data.download --config configs/download.yaml

In [ ]:
!python -m src.data.transform --config configs/transform.yaml

## 2. Train the model

The main model is the HTGNN checkpoint explained later. Training writes a checkpoint to `checkpoints/` and run metadata to `runs/`.

In [ ]:
!python -m src.train --config configs/models/pointwise/HTGNN.yaml

## 3. Run the backtest

The backtest applies portfolio rebalancing strategies to the latest trained checkpoint and stores portfolio values, allocation summaries, and metrics under `backtests/`.

In [ ]:
!python -m src.backtest --checkpoint latest --config configs/backtest.yaml

## 4. Visualise backtest results

The helper below finds the most recent backtest directory so the plots can be displayed without hard-coding timestamps.

In [ ]:
from pathlib import Path
from IPython.display import SVG, display


def latest_dir(root):
    root = Path(root)
    candidates = [path for path in root.iterdir() if path.is_dir()]
    return max(candidates, key=lambda path: path.stat().st_mtime)


backtest_dir = latest_dir("backtests")
backtest_dir

The accumulated-returns plot compares the model-driven strategy against enabled benchmarks. It is the main performance view for the trained allocation model.

In [ ]:
display(SVG(filename=str(backtest_dir / "accumulated_returns.svg")))

The allocation summary shows how the rebalancing strategy distributes capital across currencies on average. Large error bars indicate unstable or regime-dependent allocations.

In [ ]:
display(SVG(filename=str(backtest_dir / "allocations_summary.svg")))

## 5. Run the XAI pipeline

The XAI pipeline explains the latest checkpoint with the figures that are currently produced by `xai.py`: portfolio-signal SHAP, Integrated Gradients, temporal occlusion, and deletion/insertion checks. The SHAP step uses the full `portfolio_signal` lookback window and aggregates the temporal contributions by input currency.

In [ ]:
!python -m xai.xai --checkpoint latest --model auto --split test --target variance --output-dir xai --device cpu --max-samples 260 --background-samples 32 --n-local 3 --top-k 5 --seed 42 --methods all

The first SHAP grid explains the model's predicted portfolio weights, labelled as `mean` because they are the mean allocation implied by the model output. Each subplot fixes one output currency and the bars show the average absolute contribution of each input currency in `portfolio_signal`; larger bars mean that currency is more influential for that allocation decision.

In [ ]:
display(SVG(filename="xai/global_explanations/portfolio_signal_shap_mean.svg"))

The signed SHAP version keeps the direction of the average contribution. Values close to zero do not mean the input is irrelevant; they can also mean that positive and negative effects cancel over time. This view is useful for discussing net directional tendencies, while the absolute grid above is better for ranking importance.

In [ ]:
display(SVG(filename="xai/global_explanations/portfolio_signal_shap_mean_signed.svg"))

The variance SHAP grid repeats the same analysis for the model's marginal allocation uncertainty. Here the absolute bars show which input currencies most affect the predicted uncertainty for each output currency, rather than the allocation level itself.

In [ ]:
display(SVG(filename="xai/global_explanations/portfolio_signal_shap_variance.svg"))

The signed variance grid shows whether each input currency tends to increase or decrease the predicted uncertainty on average. As with the signed mean view, small net values should be interpreted as directional cancellation, not necessarily as lack of importance.

In [ ]:
display(SVG(filename="xai/global_explanations/portfolio_signal_shap_variance_signed.svg"))

Integrated Gradients attributes the selected local model output back to the heterogeneous graph inputs. The node-importance histogram summarises the average attribution per node, while the graph view places the same scores on the HTGNN topology so that influential market blocks are easier to inspect structurally.

In [ ]:
display(SVG(filename="xai/local_explanations/integrated_gradients_node_importance.svg"))

In [ ]:
display(SVG(filename="xai/local_explanations/integrated_gradients_graph.svg"))

The pie charts break the Integrated Gradients attribution down inside each observed node. They are useful when a node is important overall and we want to see whether that importance is concentrated in a few internal features or spread across the node's inputs.

In [ ]:
display(SVG(filename="xai/local_explanations/input_attention_pie_charts.svg"))

Temporal occlusion measures the performance impact of replacing each lookback window with a baseline. Large drops indicate windows whose information is useful for the strategy; small changes suggest that the model can make similar decisions without that temporal slice.

In [ ]:
display(SVG(filename="xai/local_explanations/temporal_occlusion_windows.svg"))

Node-window occlusion applies the same five temporal windows separately to each node. This avoids combinatorial perturbations and shows whether a time window matters globally or mainly through specific nodes.

In [ ]:
display(SVG(filename="xai/local_explanations/node_temporal_occlusion_windows.svg"))

Deletion/insertion is the main faithfulness sanity check. Deletion progressively removes the most important nodes from the full model input, while insertion starts from a baseline input and progressively restores those nodes. This view reports strategy performance relative to the equal-weight standard portfolio.

In [ ]:
display(SVG(filename="xai/evaluation/deletion_insertion_curve.svg"))

The second deletion/insertion view uses the same perturbation order but compares performance against the full, unperturbed model. It is stricter: it asks how much of the trained model's own performance is retained after removing or restoring the selected nodes.

In [ ]:
display(SVG(filename="xai/evaluation/deletion_insertion_vs_full_model_curve.svg"))